# Naive treatment of wetting and drying

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import dolfinx as dx
import dolfinx.fem.petsc
import dolfinx.plot as dxp
import fenicsxtools as ft
from mpi4py import MPI
import numpy as np
from petsc4py import PETSc
import pyvista as pv
import ufl
import ufl.formatting.ufl2unicode

In [3]:
pv.set_jupyter_backend('static')
plotter = pv.Plotter()

## Problem Parameters

In [4]:
# Equation parameters
def vel(x):
    return np.vstack((
        np.where( # x component
            x[1] < 0.5, # y < 0.5
            1.0 - 2 * x[1], # rightward
            0.0 # otherwise, not moving
        ),
        np.zeros(x.shape[1]) # y component is 0
    ))

In [6]:
# Initial condition
x0 = 0.5
y0 = 0.5
r0 = 0.1
def iota0(x):
    return np.exp(-((x[0] - x0) ** 2 + (x[1] - y0) ** 2)/ (2 * r0 ** 2))

## Discretization Parameters

In [7]:
# Numerical parameters
nx = 100 # Cell resolution
dt = 0.001 # Time step
nt = 1000
ntps = 1 / dt
write_every = 10 # Only every nth frame
plot_speedup = 1 # Relative to sim time

In [8]:
# Define a simple
domain = dx.mesh.create_unit_square(MPI.COMM_WORLD, nx, nx, cell_type=dx.mesh.CellType.triangle)

## FEM Formulation

In [9]:
# Define function spaces
V = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1))
W = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1, (domain.geometry.dim,)))

In [10]:
# Initialize tracer
iota = dx.fem.Function(V, name='iota')
iota.interpolate(iota0)

In [11]:
# Initialize velocity
v = dx.fem.Function(W, name='v')
v.interpolate(vel)

In [12]:
# Define equation
equation = ft.equations.get_advection(
    domain=domain,
    U=iota,
    v=v
)

In [13]:
# Choose formulation to build weak forms
system = ft.formulations.SemiDiscreteSystem.using_DG(
    equation=equation,
    trace_function=ft.fluxes.fluxn_llf_scalar
)

In [14]:
# Build an ODE solver
solver = ft.solvers.TSSolver.from_SemiDiscreteSystem(
    system=system,
    method='explicit'
)
opts = PETSc.Options()
opts['ts_ssp_type'] = 'rks3' 
solver.ts.setType(PETSc.TS.Type.SSP)
solver.ts.setTime(0.0)
solver.ts.setTimeStep(dt)
solver.ts.setMaxTime(dt * nt)
solver.ts.setExactFinalTime(PETSc.TS.ExactFinalTime.MATCHSTEP)

In [15]:
# Set up plotting
topology, cell_types, geometry = dxp.vtk_mesh(V)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)
grid.point_data['iota'] = np.asarray(iota.x.array.real).copy()
plotter = pv.Plotter()
plotter.add_mesh(
    grid,
    scalars='iota',
    cmap='jet',
    clim=[0.0, 1.0],
    show_edges=False
)
plotter.view_xy()
plotter.open_gif(
    'naive.gif',
    fps=(plot_speedup * ntps / write_every)
)

In [16]:
# Set monitor for time stepping loop
def monitor(ts, step, t, u):
    if step % write_every != 0:
        return
    # Make sure solution is updated
    dx.fem.petsc.assign(u, iota)
    iota.x.scatter_forward()
    # Plotting from rank 0
    if domain.comm.rank == 0:
        grid.point_data['iota'] = np.asarray(iota.x.array.real).copy()
        plotter.write_frame()
solver.ts.setMonitor(monitor)

In [17]:
# Finalize setup
solver.ts.setFromOptions()

## Solution Loop

In [18]:
# Solution loop
solver.ts.solve(iota.x.petsc_vec)

In [19]:
# Close and save file
plotter.close()